# Task 1 — Spell Checker Project
**Dataset:** [ArXiv Dataset](https://www.kaggle.com/datasets/Cornell-University/arxiv)

**Before running:** download `ArXiv Dataset.json` from the Kaggle link above and place it in the same folder as this notebook.

# Import libraries and configure dataset, input limits, and word-tokenization settings

In [1]:
# Import required liberies

import json, re, math
from collections import Counter, defaultdict
import tkinter as tk
from tkinter import ttk, messagebox

# Path to arXiv dataset
ARXIV_JSON_PATH = "arxiv-metadata-oai-snapshot.json"

# Limit dataset size for faster loading
MAX_ROWS = 200000

# Limit text editor input
MAX_EDITOR_CHARS = 500

# Regex pattern to extract alphabetic words
TOKEN_PATTERN = re.compile(r"[A-Za-z]+")


# Tokenization, Processing and Prepairing 


In [2]:
# TOKENIZATION
# Extract alphabetic sequences only
# Convert everything to lowercase

def tokenize_with_positions(text):
    tokens = []          # lowercase tokens for NLP
    originals = []       # original words for GUI
    positions = []       # (start, end) positions for highlighting

    for match in TOKEN_PATTERN.finditer(text):
        start, end = match.span()
        word = match.group()

        tokens.append(word.lower())      # used lowercase for NLP model
        originals.append(word)           # GUI keeps original casing
        positions.append((start, end))   # highlight exact characters  

    return tokens, originals, positions


# Minimum Edit Distance

In [3]:
# MINIMUM EDIT DISTANCE (Damerau–Levenshtein - used for generating candidate corrections for misspelled words.)

def damerau_levenshtein(s1, s2):
    len1, len2 = len(s1), len(s2)
    INF = len1 + len2
    da = {}
    dist = [[0]*(len2+2) for _ in range(len1+2)]
    dist[0][0] = INF

    # Initialize boundaries
    for i in range(len1+1):
        dist[i+1][0] = INF
        dist[i+1][1] = i
    for j in range(len2+1):
        dist[0][j+1] = INF
        dist[1][j+1] = j

    # Main DP loop
    for i in range(1, len1+1):
        db = 0
        for j in range(1, len2+1):
            i1 = da.get(s2[j-1], 0)
            j1 = db
            cost = 0 if s1[i-1] == s2[j-1] else 1
            if cost == 0:
                db = j

            dist[i+1][j+1] = min(
                dist[i][j] + cost,
                dist[i+1][j] + 1,
                dist[i][j+1] + 1,
                dist[i1][j1] + (i-i1-1) + 1 + (j-j1-1)
            )

        da[s1[i-1]] = i

    return dist[len1+1][len2+1]


# Probabilistic Language Models

In [4]:
# LANGUAGE MODEL
# Trained a simple statistical language model from the ArXiv dataset:
# Unigram counts
# Bigram counts

class LanguageModel:
    def __init__(self):
        self.unigram = Counter()
        self.bigram = defaultdict(Counter)
        self.vocab = set()
        self.total = 0
        self.V = 0
        self.length_buckets = defaultdict(list)

    def train_from_json(self, path, max_rows):
        print("Loading JSON...")
        count = 0

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                abstract = str(row.get("abstract", ""))
                tokens, _, _ = tokenize_with_positions(abstract)

                if tokens:
                    self.unigram.update(tokens)
                    self.total += len(tokens)
                    self.vocab.update(tokens)

                    # Build bigram counts
                    for i in range(len(tokens)-1):
                        self.bigram[tokens[i]][tokens[i+1]] += 1

                count += 1
                if count >= max_rows:
                    break

        self.V = len(self.vocab)

        # Bucket words by length for faster edit-distance search
        for w in self.vocab:
            self.length_buckets[len(w)].append(w)

        print(f"Corpus built: {self.total} tokens, {self.V} vocab words.")

    def unigram_prob(self, w):
        return (self.unigram[w] + 1) / (self.total + self.V)

    def bigram_prob(self, w1, w2):
        if w1 is None or w2 is None:
            return 1.0 / (self.total + self.V)
        return (self.bigram[w1][w2] + 1) / (self.unigram[w1] + self.V)


# Check Spelling of the Input

In [5]:
# SPELL CHECKER
# Non-word errors (misspellings)
# Real-word errors (valid words used incorrectly)

class SpellChecker:
    def __init__(self, lm):
        self.lm = lm

        # Common confusion pairs
        self.real_word_confusions = {
            "affect": "effect",
            "week": "weak",
            "bass": "base",
            "to": "too"
        }

        # Grammar rule: "too + adjective"
        self.adjective_list = {
            "high", "low", "small", "large", "big", "weak", "strong",
            "slow", "fast", "wide", "narrow", "short", "long",
            "heavy", "light", "deep", "shallow"
        }

    # Generate candidate corrections using edit distance
    def generate_candidates(self, word):
        L = len(word)
        candidates = []

        for length in range(L-2, L+3):
            if length in self.lm.length_buckets:
                for w in self.lm.length_buckets[length]:
                    d = damerau_levenshtein(word, w)
                    if d <= 2:
                        candidates.append((w, d))

        return candidates

    # Score candidate using unigram + bigram + edit distance
    def score(self, cand, dist, prev, nxt):
        uni = self.lm.unigram_prob(cand)
        p_prev = self.lm.bigram_prob(prev, cand)
        p_next = self.lm.bigram_prob(cand, nxt)
        return math.log(uni) + math.log(p_prev) + math.log(p_next) - dist

    # Suggest top 5 corrections
    def suggest(self, word, prev, nxt):
        cands = self.generate_candidates(word)
        scored = [(c, d, self.score(c, d, prev, nxt)) for c, d in cands]
        scored.sort(key=lambda x: x[2], reverse=True)
        return scored[:5]

    # Detect real-word errors using context
    def detect_real_word_context_based(self, tokens):
        errors = []

        for i, w in enumerate(tokens):
            lw = w.lower()

            if lw not in self.real_word_confusions:
                continue

            prev = tokens[i-1] if i > 0 else None
            nxt = tokens[i+1] if i < len(tokens)-1 else None

            # Grammar rule: "to + adjective" → "too"
            if lw == "to" and nxt in self.adjective_list:
                errors.append({
                    "index": i,
                    "word": w,
                    "type": "real-word",
                    "suggestions": [("too", 1, 999)]
                })
                continue

            # Bigram context scoring
            actual_score = (
                math.log(self.lm.unigram_prob(lw)) +
                math.log(self.lm.bigram_prob(prev, lw)) +
                math.log(self.lm.bigram_prob(lw, nxt))
            )

            alt = self.real_word_confusions[lw]
            alt_score = (
                math.log(self.lm.unigram_prob(alt)) +
                math.log(self.lm.bigram_prob(prev, alt)) +
                math.log(self.lm.bigram_prob(alt, nxt))
            )

            if alt_score > actual_score + 1.0:
                errors.append({
                    "index": i,
                    "word": w,
                    "type": "real-word",
                    "suggestions": [(alt, 1, alt_score)]
                })

        return errors

    # Main error detection
    def detect(self, tokens):
        errors = {}

        # Non-word errors
        for i, w in enumerate(tokens):
            prev = tokens[i-1] if i > 0 else None
            nxt = tokens[i+1] if i < len(tokens)-1 else None

            if w not in self.lm.vocab:
                errors[i] = {
                    "index": i,
                    "word": w,
                    "type": "non-word",
                    "suggestions": self.suggest(w, prev, nxt)
                }

        # Real-word errors
        real_errors = self.detect_real_word_context_based(tokens)
        for err in real_errors:
            errors[err["index"]] = err

        return [errors[i] for i in sorted(errors.keys())]


# GUI

In [6]:
# TKINTER GUI

class SpellCheckerGUI:
    def __init__(self, root, checker, lm):
        self.root = root
        self.checker = checker
        self.lm = lm

        self.errors = []
        self.token_positions = []
        self.original_tokens = []
        self.selected_error = None

        self.root.title("Scientific Spell Checker")
        self.build_gui()

    def build_gui(self):
        main = ttk.Frame(self.root)
        main.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        left = ttk.Frame(main)
        left.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        ttk.Button(left, text="Check Spelling", command=self.check_spelling).pack(anchor=tk.W)
        ttk.Button(left, text="Replace All", command=self.replace_all).pack(anchor=tk.W, pady=(5, 5))

        ttk.Label(left, text="Text Editor (500 characters):").pack(anchor=tk.W)

        self.text = tk.Text(left, height=15, width=50)
        self.text.pack(fill=tk.BOTH, expand=True)
        self.text.bind("<KeyRelease>", self.limit_chars)
        self.text.bind("<Button-1>", self.on_click)

        # Color-coded highlighting
        self.text.tag_configure("nonword", foreground="red", underline=True)
        self.text.tag_configure("realword", foreground="blue", underline=True)

        ttk.Label(left, text="Suggestions:").pack(anchor=tk.W, pady=(10, 0))
        self.suggestions = tk.Listbox(left, height=8)
        self.suggestions.pack(fill=tk.BOTH, expand=True)
        self.suggestions.bind("<<ListboxSelect>>", self.apply_suggestion)

        right = ttk.Frame(main)
        right.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(20, 0))

        ttk.Label(right, text="Dictionary:").pack(anchor=tk.W)
        self.dict_list = tk.Listbox(right, height=20)
        self.dict_list.pack(fill=tk.BOTH, expand=True)

        ttk.Label(right, text="Search:").pack(anchor=tk.W)
        self.search = ttk.Entry(right)
        self.search.pack(fill=tk.X)
        self.search.bind("<KeyRelease>", self.update_dict)

        self.load_dict()

    def load_dict(self):
        sorted_vocab = sorted(self.lm.vocab)
        self.full_vocab = sorted_vocab
        for w in sorted_vocab:
            self.dict_list.insert(tk.END, w)

    def update_dict(self, event=None):
        q = self.search.get().lower()
        self.dict_list.delete(0, tk.END)
        for w in self.full_vocab:
            if q in w:
                self.dict_list.insert(tk.END, w)

    def limit_chars(self, event=None):
        content = self.text.get("1.0", tk.END)
        if len(content) > MAX_EDITOR_CHARS + 1:
            self.text.delete("1.0", tk.END)
            self.text.insert("1.0", content[:MAX_EDITOR_CHARS])

    def check_spelling(self):
        content = self.text.get("1.0", tk.END).strip()
        self.text.tag_remove("nonword", "1.0", tk.END)
        self.text.tag_remove("realword", "1.0", tk.END)
        self.suggestions.delete(0, tk.END)

        tokens, originals, positions = tokenize_with_positions(content)
        self.original_tokens = originals
        self.token_positions = positions

        self.errors = self.checker.detect(tokens)

        if not self.errors:
            messagebox.showinfo("OK", "No spelling errors detected.")
            return

        self.highlight()

    def highlight(self):
       self.error_positions = {}

       for err in self.errors:
           start, end = self.token_positions[err["index"]]

           tag = "nonword" if err["type"] == "non-word" else "realword"
           self.text.tag_add(tag, f"1.0+{start}c", f"1.0+{end}c")

           self.error_positions[(start, end)] = err["index"]


           self.error_positions[(start, end)] = err["index"]

    def on_click(self, event=None):
        index = self.text.index(f"@{event.x},{event.y}")
        abs_pos = int(self.text.count("1.0", index, "chars")[0])

        for (start, end), err_idx in self.error_positions.items():
            if start <= abs_pos < end:
                self.show_suggestions(err_idx)
                return

    def apply_suggestion(self, event=None):
        sel = self.suggestions.curselection()
        if not sel:
            return

        chosen = self.suggestions.get(sel[0]).split(" ")[0]

        # Find the error object by token index
        err_obj = None
        for e in self.errors:
             if e["index"] == self.selected_error:
                err_obj = e
                break

        if err_obj is None:
            return

        idx = err_obj["index"]
        original_word = self.original_tokens[idx]

        if original_word.istitle():
            chosen = chosen.capitalize()
        elif original_word.isupper():
              chosen = chosen.upper()

        start, end = self.token_positions[idx]
        content = self.text.get("1.0", tk.END)
        new_text = content[:start] + chosen + content[end:]

        self.text.delete("1.0", tk.END)
        self.text.insert("1.0", new_text)
        self.check_spelling()

    def show_suggestions(self, err_idx):
        self.selected_error = err_idx
        self.suggestions.delete(0, tk.END)

        # Find the correct error object by token index
        err_obj = None
        for e in self.errors:
            if e["index"] == err_idx:
                err_obj = e
                break

        if err_obj is None:
            return

        for cand, dist, score in err_obj["suggestions"]:
             self.suggestions.insert(tk.END, f"{cand} (distance={dist})")



    def replace_all(self):
        if not self.errors:
            messagebox.showinfo("OK", "No errors to replace.")
            return

        content = self.text.get("1.0", tk.END)

        for err in reversed(self.errors):
             idx = err["index"]
             start, end = self.token_positions[idx]

             cand = err["suggestions"][0][0]

             original_word = self.original_tokens[idx]
             if original_word.istitle():
                 cand = cand.capitalize()
             elif original_word.isupper():
                  cand = cand.upper()

             content = content[:start] + cand + content[end:]

             self.text.delete("1.0", tk.END)
             self.text.insert("1.0", content)
             self.check_spelling()


# Main Function

In [7]:
# MAIN PROGRAM - this cell launches the full application.

def main():
    lm = LanguageModel()
    lm.train_from_json(ARXIV_JSON_PATH, MAX_ROWS)

    checker = SpellChecker(lm)

    root = tk.Tk()
    SpellCheckerGUI(root, checker, lm)
    root.mainloop()

# Run the application
main()


Loading JSON...
Corpus built: 25407656 tokens, 116718 vocab words.
